In [1]:
# Dependencies
# This program isolates a likely bubble-plume vertical velocity signal from ADCP data,
# compares that signal to bottom-pressure data, and optionally removes the pressure-linked component.
# The workflow includes:
# - loading ADCP upward velocity data
# - estimating a depth-dependent background vertical velocity
# - isolating the plume anomaly at a selected near-bottom depth
# - optionally computing a plume-core proxy from the bottom window
# - loading and aligning bottom-pressure data
# - optionally applying a linear pressure correction
# - plotting summary figures of the raw, corrected, and normalized signals

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
# SETTINGS

# File paths for the ADCP velocity dataset and the bottom-pressure dataset

ADCP_NC = r"C:\Users\chero\Downloads\RS01SUM2-MJ01B-12-ADCPSK101_hourly_20140918220000-20171220211415.nc"
PRES_NC = r"C:\Users\chero\Downloads\ooi-rs01sum1-lj01b-09-prestb102_6609_cfcb_b907.nc"

# USER SETTINGS

# Choose the analysis depth as an offset above the maximum ADCP z value.
# The plume anomaly is computed at this target depth relative to a depth-based background.

DEPTH_OFFSET_ABOVE_MAX_M = 20.0

# Background estimate for vertical water velocity
# The background is computed over depth using either the median or mean at each time step.
BACKGROUND_METHOD = "median"  # options: "median", "mean"

# Optional QC masking
# If enabled, any ADCP upward velocity values flagged as QC failures are masked before analysis.
MASK_WHEN_QC_FAIL_GT0 = False

# Optional smoothing for plotting
# Apply a rolling median in hours to reduce short-timescale noise in the plotted time series.
ROLLING_HOURS = 0  # e.g. 6 or 24, set 0 to disable

# Optional plume-core proxy
# If enabled, compute the maximum positive plume anomaly within the bottom window of the ADCP profile.
COMPUTE_PLUME_CORE_PROXY = True
PLUME_CORE_WINDOW_M = 60.0

# Plot thinning
# STRIDE reduces plotting density by taking every nth point.
STRIDE = 1

# Pressure-correction settings
# These settings control whether bottom pressure is used to estimate and remove a pressure-linked signal.
USE_PRESSURE_CORRECTION = True
PRESSURE_RESAMPLE = "1H"
PRESSURE_DETREND = False

In [3]:
# HELPERS

# Helper utilities for:
# - finding variables in xarray datasets
# - smoothing time series
# - detrending pressure
# - enforcing a usable time dimension
# - printing compact summary statistics

def pick_first_existing(container, candidates):
    for name in candidates:
        if name in container:
            return name
    return None


def pick_velocity_var(ds: xr.Dataset) -> str:
    candidates = [
        "upward_sea_water_velocity_mean",
        "upward_sea_water_velocity",
        "w",
    ]
    name = pick_first_existing(ds.data_vars, candidates)
    if name is not None:
        return name
    raise KeyError(
        "Could not find an upward velocity variable. "
        f"Available vars include: {list(ds.data_vars)[:40]}"
    )


def pick_pressure_var(ds: xr.Dataset) -> str:
    candidates = [
        "seawater_pressure",
        "bottom_pressure",
        "pressure",
        "int_ctd_pressure",
        "abs_seafloor_pressure",
        "seafloor_pressure",
        "PRES",
    ]
    name = pick_first_existing(ds.data_vars, candidates)
    if name is not None:
        return name

    for name, da in ds.data_vars.items():
        long_name = str(da.attrs.get("long_name", "")).lower()
        std_name = str(da.attrs.get("standard_name", "")).lower()
        units = str(da.attrs.get("units", "")).lower()

        if ("pressure" in long_name) or ("pressure" in std_name):
            return name
        if units in ["dbar", "bar", "pa", "kpa", "mpa"]:
            if "pres" in name.lower() or "press" in name.lower():
                return name

    raise KeyError(
        "Could not find a pressure variable. "
        f"Available vars include: {list(ds.data_vars)[:40]}"
    )


def rolling_median(da: xr.DataArray, hours: int) -> xr.DataArray:
    if hours and hours > 0:
        return da.rolling(
            time=hours,
            center=True,
            min_periods=max(2, hours // 3)
        ).median()
    return da


def detrend_linear(da: xr.DataArray) -> xr.DataArray:
    y = da.values.astype(float)
    mask = np.isfinite(y)

    if mask.sum() < 10:
        return da

    x = np.arange(y.size)
    coeffs = np.polyfit(x[mask], y[mask], 1)
    y_detrended = y - (coeffs[0] * x + coeffs[1])

    return xr.DataArray(
        y_detrended,
        coords=da.coords,
        dims=da.dims,
        attrs=da.attrs
    )


def ensure_time_dimension(ds: xr.Dataset, da: xr.DataArray) -> xr.DataArray:
    """
    Convert pressure data indexed by something like ('row',)
    into a time-indexed DataArray with dim ('time',).
    """
    if "time" in da.dims:
        return da

    if "time" in ds.variables and "row" in da.dims:
        ds = ds.assign_coords(time=("row", ds["time"].values))
        return ds[da.name].swap_dims({"row": "time"}).drop_vars("row", errors="ignore")

    if "row" in da.dims:
        row_n = ds.dims.get("row", None)
        time_candidate = None

        for vname, v in ds.variables.items():
            if vname.lower() in ("time", "timestamp", "date_time") and v.dims == ("row",):
                time_candidate = vname
                break

        if time_candidate is None and row_n is not None:
            for vname, v in ds.variables.items():
                if v.dims == ("row",) and np.issubdtype(v.dtype, np.datetime64):
                    time_candidate = vname
                    break

        if time_candidate is None:
            raise ValueError(
                f"Pressure has dims {da.dims} but no usable time variable was found. "
                f"Dataset variables include: {list(ds.variables)[:40]}"
            )

        ds = ds.assign_coords(time=("row", ds[time_candidate].values))
        return ds[da.name].swap_dims({"row": "time"}).drop_vars("row", errors="ignore")

    raise ValueError(f"Pressure data has dims {da.dims} and no obvious way to create a time dimension.")


def print_stats_cms(da_cms: xr.DataArray, name: str):
    arr = da_cms.values
    arr = arr[np.isfinite(arr)]

    if arr.size == 0:
        print(f"{name}: no finite data")
        return

    p50 = np.nanpercentile(arr, 50)
    p90 = np.nanpercentile(arr, 90)
    p99 = np.nanpercentile(arr, 99)
    vmax = np.nanmax(arr)

    print(f"{name}: median={p50:.3f} cm/s | p90={p90:.3f} | p99={p99:.3f} | max={vmax:.3f}")

In [5]:
# LOAD ADCP DATA AND COMPUTE PLUME ANOMALY

# Open the ADCP dataset, identify the upward velocity variable,
# select a near-bottom target depth, estimate the depth-based background velocity,
# and compute the plume-isolated anomaly as:
# w_plume(t) = w(z_target, t) - w0(t)

dsA = xr.open_dataset(ADCP_NC)

vel_name = pick_velocity_var(dsA)
vel = dsA[vel_name]

if "z" not in vel.dims:
    raise ValueError(f"Velocity var '{vel_name}' has dims={vel.dims}; expected to include 'z'.")

z = dsA["z"]
tA = dsA["time"]

if MASK_WHEN_QC_FAIL_GT0 and "upward_sea_water_velocity_qc_fail" in dsA:
    qc_fail = dsA["upward_sea_water_velocity_qc_fail"]
    vel = vel.where(qc_fail == 0)

z_max = float(z.max().values)
z_target = z_max - float(DEPTH_OFFSET_ABOVE_MAX_M)

vel_at_target = vel.sel(z=z_target, method="nearest")
z_chosen = float(vel_at_target["z"].values)

if BACKGROUND_METHOD.lower() == "median":
    w0 = vel.median(dim="z", skipna=True)
elif BACKGROUND_METHOD.lower() == "mean":
    w0 = vel.mean(dim="z", skipna=True)
else:
    raise ValueError("BACKGROUND_METHOD must be 'median' or 'mean'.")

w_plume = vel_at_target - w0

# Optional smoothing
# Apply rolling median smoothing to the background, target-bin, and plume-anomaly time series.
w0_s = rolling_median(w0, ROLLING_HOURS)
w_target_s = rolling_median(vel_at_target, ROLLING_HOURS)
w_plume_s = rolling_median(w_plume, ROLLING_HOURS)

# Convert to cm/s
# All printed statistics and plotted figures below use cm/s instead of m/s.
to_cms = 100.0
w0_cms = w0_s * to_cms
w_target_cms = w_target_s * to_cms
w_plume_cms = w_plume_s * to_cms

print("\nADCP SUMMARY")
print(f"Velocity variable: {vel_name}")
print(f"Max depth (z_max): {z_max:.3f} m")
print(f"Requested target depth: z_max - {DEPTH_OFFSET_ABOVE_MAX_M:.1f} m = {z_target:.3f} m")
print(f"Chosen nearest z bin: {z_chosen:.3f} m")
print(f"Computed plume anomaly as w_plume(t) = w(z_target,t) - depth-{BACKGROUND_METHOD}")

ValueError: found the following matches with the input file in xarray's IO backends: ['netcdf4', 'h5netcdf']. But their dependencies may not be installed, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

In [6]:
# PLUME-CORE PROXY

# Compute a plume-core proxy by taking the maximum positive anomaly
# in the bottom PLUME_CORE_WINDOW_M of the ADCP profile at each time step.

w_core_cms = None
z_core_min = None

if COMPUTE_PLUME_CORE_PROXY:
    z_core_min = z_max - float(PLUME_CORE_WINDOW_M)
    vel_bottom = vel.where((z >= z_core_min) & (z <= z_max), drop=True)
    w_anom_bottom = vel_bottom - w0
    w_core = w_anom_bottom.max(dim="z", skipna=True)
    w_core = w_core.where(w_core > 0)
    w_core_s = rolling_median(w_core, ROLLING_HOURS)
    w_core_cms = w_core_s * to_cms

NameError: name 'z_max' is not defined

In [7]:
# LOAD PRESSURE DATA, FIX DIMENSIONS, AND ALIGN TO ADCP TIME

# Open the pressure dataset, identify the pressure variable,
# ensure that it has a usable time dimension, resample it to the requested interval,
# optionally detrend it, and then align it to the ADCP plume-anomaly time series.

dsP = xr.open_dataset(PRES_NC)

pres_name = pick_pressure_var(dsP)
P = dsP[pres_name]

print("\nPRESSURE SUMMARY")
print(f"Pressure variable: {pres_name}")
print(f"Original dims: {P.dims}")
print(f"Units: {P.attrs.get('units', '(unknown)')}")

P = ensure_time_dimension(dsP, P)

if not np.issubdtype(P["time"].dtype, np.datetime64):
    dsP = xr.decode_cf(dsP)
    P = dsP[pres_name]
    P = ensure_time_dimension(dsP, P)

print(f"Pressure after time fix | dims: {P.dims} | time dtype: {P['time'].dtype}")

P_h = P.resample(time=PRESSURE_RESAMPLE).median()

if PRESSURE_DETREND:
    P_h = detrend_linear(P_h)

# Align overlapping times
# Keep only time steps shared by both the plume anomaly and pressure records.
w_aligned, P_aligned = xr.align(w_plume_s, P_h, join="inner")

good = np.isfinite(w_aligned.values) & np.isfinite(P_aligned.values)
good_idx = np.where(good)[0]

w_aligned = w_aligned.isel(time=good_idx)
P_aligned = P_aligned.isel(time=good_idx)


PRESSURE SUMMARY
Pressure variable: sea_water_pressure_at_sea_floor
Original dims: ('row',)
Units: decibars
Pressure after time fix | dims: ('time',) | time dtype: datetime64[ns]


C:\Users\chero\Anaconda\Lib\site-packages\xarray\groupers.py:530: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  self.index_grouper = pd.Grouper(


NameError: name 'w_plume_s' is not defined

In [ ]:
# LINEAR PRESSURE CORRECTION

# Fit a linear relation between plume anomaly and bottom pressure,
# then remove the pressure-linked component from the plume signal:
# w_corr = w_plume - beta * (P - mean(P))

beta = np.nan
w_corr = None
w_corr_cms = None

if USE_PRESSURE_CORRECTION and w_aligned.size > 20:
    x = P_aligned.values.astype(float)
    y = w_aligned.values.astype(float)

    x0 = np.nanmean(x)
    beta = np.polyfit(x - x0, y, 1)[0]

    w_corr = w_aligned - beta * (P_aligned - x0)
    w_corr_cms = rolling_median(w_corr, ROLLING_HOURS) * to_cms

    print("\nPRESSURE CORRECTION")
    print(f"beta = {beta:.4e} (m/s) per {P_aligned.attrs.get('units', 'pressure_unit')}")
else:
    print("\nPressure correction skipped (disabled or insufficient overlap).")

In [ ]:
# SUMMARY STATISTICS

# Print compact summary statistics for the target-bin velocity, background velocity,
# plume-isolated anomaly, optional plume-core proxy, and optional pressure-corrected anomaly.

print("\n--- SUMMARY STATS (cm/s) ---")
print_stats_cms(w_target_cms, f"Raw w at z≈{z_chosen:.1f} m")
print_stats_cms(w0_cms, "Background w0(t)")
print_stats_cms(w_plume_cms, "Plume-isolated w_plume(t) = w(z_target)-w0")

if w_core_cms is not None:
    print_stats_cms(
        w_core_cms,
        f"Plume-core proxy (bottom {PLUME_CORE_WINDOW_M:.0f} m max anomaly, clipped >0)"
    )

if w_corr_cms is not None:
    print_stats_cms(
        w_corr_cms,
        "Pressure-corrected plume anomaly"
    )

In [ ]:
# FIGURE 1: Raw Target-Bin, Background, and Plume-Isolated Velocity

# Plot the near-bottom target-bin velocity, the depth-based background velocity,
# and the plume-isolated anomaly through time.

tt = tA.values[::STRIDE]

plt.figure(figsize=(14, 5))
plt.plot(tt, w_target_cms.values[::STRIDE], linewidth=0.8, label=f"Raw w(z≈{z_chosen:.1f} m)")
plt.plot(tt, w0_cms.values[::STRIDE], linewidth=1.0, label=f"Background w0(t) ({BACKGROUND_METHOD})")
plt.plot(tt, w_plume_cms.values[::STRIDE], linewidth=1.0, label="Plume-isolated w_plume(t)")
plt.axhline(0, linewidth=1)

title = (
    f"Bubble-plume isolation from ADCP upward velocity\n"
    f"Target bin: z≈{z_chosen:.1f} m (max(z)-{DEPTH_OFFSET_ABOVE_MAX_M:.0f} m). "
    f"w_plume = w(z_target) - w0(t), w0 = {BACKGROUND_METHOD} over depth"
)
if ROLLING_HOURS and ROLLING_HOURS > 0:
    title += f" | {ROLLING_HOURS}h rolling median applied"

plt.title(title)
plt.xlabel("Time")
plt.ylabel("Upward velocity (cm/s)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# FIGURE 2: Plume-Core Proxy

# Plot the maximum positive anomaly within the bottom analysis window,

if w_core_cms is not None:
    plt.figure(figsize=(14, 4))
    plt.plot(tt, w_core_cms.values[::STRIDE], linewidth=1.0)
    plt.axhline(0, linewidth=1)
    plt.title(
        f"Plume-core strength proxy: max positive anomaly in bottom {PLUME_CORE_WINDOW_M:.0f} m\n"
        f"w_core(t) = max_z[w(z,t)-w0(t)] (clipped >0), window z∈[{z_core_min:.1f}, {z_max:.1f}] m"
    )
    plt.xlabel("Time")
    plt.ylabel("Core anomaly (cm/s)")
    plt.tight_layout()
    plt.show()

In [ ]:
# FIGURE 3: Bottom Pressure Time Series

# Plot the bottom-pressure record after resampling to the selected interval.

plt.figure(figsize=(14, 4))
plt.plot(P_h["time"].values[::STRIDE], P_h.values[::STRIDE], linewidth=0.8)
plt.title("Bottom pressure (resampled to hourly)")
plt.xlabel("Time")
plt.ylabel(f"Pressure ({P_h.attrs.get('units', 'unknown')})")
plt.tight_layout()
plt.show()

In [ ]:
# FIGURE 4: Plume-Isolated and Pressure-Corrected Velocity Time Series

# Compare the original plume-isolated anomaly to the pressure-corrected anomaly,
# if the pressure correction was successfully computed.

plt.figure(figsize=(14, 5))
plt.plot(
    w_plume_cms["time"].values[::STRIDE],
    w_plume_cms.values[::STRIDE],
    linewidth=0.9,
    label=f"Plume-isolated vertical velocity (w(z≈{z_chosen:.1f} m) − depth-{BACKGROUND_METHOD})"
)

if w_corr_cms is not None:
    plt.plot(
        w_corr_cms["time"].values[::STRIDE],
        w_corr_cms.values[::STRIDE],
        linewidth=1.0,
        label="Pressure-corrected plume anomaly (pressure-linked component removed)"
    )

plt.axhline(0, linewidth=1)
plt.title("Bubble-plume vertical velocity anomalies (cm/s)")
plt.xlabel("Time")
plt.ylabel("Vertical velocity anomaly (cm/s)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# FIGURE 5: Plume Anomaly vs Bottom Pressure
# Scatter plot comparing the plume-isolated anomaly to bottom pressure
# over the overlapping time interval.

plt.figure(figsize=(6.5, 5.5))
plt.scatter(P_aligned.values, w_aligned.values * 100.0, s=6, alpha=0.35)
plt.title("Plume-isolated vertical velocity vs bottom pressure")
plt.xlabel(f"Bottom pressure ({P_aligned.attrs.get('units', 'unknown')})")
plt.ylabel("Plume-isolated vertical velocity (cm/s)")
plt.tight_layout()
plt.show()

In [ ]:
# FIGURE 6: Normalized Pressure and Plume Comparison
# Normalize both bottom pressure and plume anomaly as z-scores
# so their relative timing and covariance can be compared directly.

wp = w_aligned.values * 100.0
pp = P_aligned.values.astype(float)

wp_n = (wp - np.nanmean(wp)) / (np.nanstd(wp) + 1e-12)
pp_n = (pp - np.nanmean(pp)) / (np.nanstd(pp) + 1e-12)

plt.figure(figsize=(14, 4))
plt.plot(w_aligned["time"].values[::STRIDE], wp_n[::STRIDE], linewidth=0.9, label="w_plume (normalized)")
plt.plot(P_aligned["time"].values[::STRIDE], pp_n[::STRIDE], linewidth=0.9, label="Pressure (normalized)")
plt.axhline(0, linewidth=1)
plt.title("Normalized comparison: plume anomaly vs bottom pressure (hourly, overlapping times)")
plt.xlabel("Time")
plt.ylabel("Normalized units (z-score)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# CLEANUP
# Close the ADCP and pressure datasets after plotting is complete.

dsA.close()
dsP.close()